# Radar 모범답안2 (Weighted-CE U-Net) — 실제 캐글 대회 제출 (Colab)

IOAI 2025 Individual **Radar** *모범답안2*(개선판, 0.7963→**0.9834**)의 코드를 **그대로** 옮겨, 사설 캐글 대회
[**radar-ioai-2025**](https://www.kaggle.com/competitions/radar-ioai-2025) 에서 다운로드·학습·제출한다.

**기법**: 6채널 레이더 입력을 **다중클래스 U-Net** 으로 분할하되, **가중 교차엔트로피**(배경 1, 비배경 클래스 50)로
학습 — 채점 가중과 일치시켜 희소한 비배경 클래스를 살린다(0.98 의 핵심).

**구성** (모범답안2 그대로): `load_img`(채널 정규화) · `RadarDS`(`.mat.pt`, 라벨 −1..3→0..4) · `DC`/`UNet`(cin=6,cout=5)
· `crit=CrossEntropyLoss(weight=[1,50,50,50,50])` · 50×181→56×192 패딩→U-Net→크롭.

> **제출 형식**: 대회 baseline 과 동일하게 `submission.csv`(`filename, pixel_0…pixel_N` = 픽셀별 **argmax 클래스 0..4**).
> (로컬 모범답안2 는 로컬 채점기용으로 `argmax−1`(−1..3)을 쓰지만, radar-ioai-2025 는 baseline 형식인 **argmax(0..4)** 로 채점.)
> ⚙️ GPU 필요(U-Net b=64 + AMP). T4 수 분. ⚠️ API 토큰 평문 — 외부 공유 금지.

## 0. 설치 · 자격증명 · 시드

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
import numpy as np, torch
torch.manual_seed(0); np.random.seed(0)
print("준비 완료")

## 1. Kaggle 에서 radar-ioai-2025 다운로드 (training_set / test_set)
403 이 나면 [대회 페이지](https://www.kaggle.com/competitions/radar-ioai-2025/rules)에서 'I Understand and Accept' 1회 후 재실행.

In [ ]:
import glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
try:
    api.competition_download_files("radar-ioai-2025", path=WORK_DIR, quiet=False)
    for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
        with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
        os.remove(zp)
    print("다운로드 완료")
except Exception as e:
    print("⚠️ 다운로드 실패:", repr(e)[:200])
    print("→ 403 이면 https://www.kaggle.com/competitions/radar-ioai-2025/rules 에서 수락 후 재실행")
# 압축 내부 폴더가 한 겹 더 있음: training_set/training_set, test_set/test_set
def _find(*cs):
    for c in cs:
        if os.path.isdir(c) and glob.glob(c + "/*.mat.pt"): return c
    return cs[0]
TRAIN_DIR = _find(os.path.join(WORK_DIR, "training_set/training_set"), os.path.join(WORK_DIR, "training_set"))
TEST_DIR  = _find(os.path.join(WORK_DIR, "test_set/test_set"), os.path.join(WORK_DIR, "test_set"))
print("train:", len(glob.glob(TRAIN_DIR+"/*.mat.pt")), "| test:", len(glob.glob(TEST_DIR+"/*.mat.pt")))

## 2. load_img · RadarDS (모범답안2 그대로)
`.mat.pt` 는 (7, 50, 181): `d[:6]`=6채널 입력, `d[6]`=라벨(−1..3). 채널별 정규화 후, 라벨은 +1 해 0..4 로 학습.

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
DEV = "cuda" if torch.cuda.is_available() else "cpu"

# 50x181 -> 56x192(÷8 패딩) -> U-Net -> 50x181 크롭
PAD = (5,6,3,3); RC = slice(3,53); CC = slice(5,186)
padf = lambda x: F.pad(x, PAD); cropf = lambda x: x[..., RC, CC]
def load_img(d):
    img = d[:6].float(); return (img - img.mean((1,2), keepdim=True)) / (img.std((1,2), keepdim=True) + 1e-6)

class RadarDS(Dataset):
    def __init__(s, paths, lbl=True): s.p = paths; s.lbl = lbl
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        d = torch.load(s.p[i], weights_only=True)
        if s.lbl: return load_img(d), (d[6]+1).long(), os.path.basename(s.p[i])   # 라벨 −1..3 → 0..4
        return load_img(d), 0, os.path.basename(s.p[i])

## 3. DC · UNet (모범답안2 그대로, cin=6 cout=5)

In [ ]:
class DC(nn.Module):
    def __init__(s, ci, co):
        super().__init__()
        s.n = nn.Sequential(nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(True),
                            nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(True))
    def forward(s, x): return s.n(x)

class UNet(nn.Module):
    def __init__(s, cin=6, cout=5, b=64):
        super().__init__()
        s.e1 = DC(cin, b); s.e2 = DC(b, b*2); s.e3 = DC(b*2, b*4); s.bo = DC(b*4, b*8); s.pool = nn.MaxPool2d(2)
        s.u3 = nn.ConvTranspose2d(b*8, b*4, 2, stride=2); s.d3 = DC(b*8, b*4)
        s.u2 = nn.ConvTranspose2d(b*4, b*2, 2, stride=2); s.d2 = DC(b*4, b*2)
        s.u1 = nn.ConvTranspose2d(b*2, b, 2, stride=2);   s.d1 = DC(b*2, b); s.o = nn.Conv2d(b, cout, 1)
    def forward(s, x):
        e1 = s.e1(x); e2 = s.e2(s.pool(e1)); e3 = s.e3(s.pool(e2)); bo = s.bo(s.pool(e3))
        d3 = s.d3(torch.cat([s.u3(bo), e3], 1)); d2 = s.d2(torch.cat([s.u2(d3), e2], 1)); d1 = s.d1(torch.cat([s.u1(d2), e1], 1))
        return s.o(d1)

## 4. 학습 — 가중 CE(배경 1, 비배경 50) (모범답안2의 핵심 개선)

In [ ]:
from sklearn.model_selection import train_test_split
EPOCHS = 10; BS = 16
paths = sorted(glob.glob(f"{TRAIN_DIR}/*.mat.pt"))
tr, _ = train_test_split(paths, test_size=0.1, random_state=42)
trdl = DataLoader(RadarDS(tr), batch_size=BS, shuffle=True, num_workers=2, drop_last=True)
model = UNet().to(DEV)
crit = nn.CrossEntropyLoss(weight=torch.tensor([1.,50.,50.,50.,50.], device=DEV))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS); scaler = torch.amp.GradScaler("cuda")
for ep in range(EPOCHS):
    model.train(); tot = 0
    for img, lbl, _ in trdl:
        img, lbl = img.to(DEV), lbl.to(DEV); opt.zero_grad()
        with torch.amp.autocast("cuda"): loss = crit(cropf(model(padf(img))), lbl)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item()
    sched.step(); print(f"epoch {ep+1}/{EPOCHS}  loss {tot/len(trdl):.4f}")

## 5. 제출 — test_set 예측 → submission.csv (대회 baseline 형식)
각 test `.mat.pt` 를 예측해 **픽셀별 argmax 클래스(0..4)** 를 `filename, pixel_0…pixel_N` 으로 저장(대회 형식).

In [ ]:
import pandas as pd
model.eval(); fns = []; preds = []
dl = DataLoader(RadarDS(sorted(glob.glob(f"{TEST_DIR}/*.mat.pt")), lbl=False), batch_size=BS, num_workers=2)
with torch.no_grad():
    for img, _, fn in dl:
        with torch.amp.autocast("cuda"): o = cropf(model(padf(img.to(DEV))))
        p = o.argmax(1).cpu().numpy().astype(int)              # 0..4 (대회 baseline 과 동일, −1 안 함)
        for j in range(p.shape[0]): preds.append(p[j].flatten()); fns.append(fn[j])
arr = np.stack(preds)
df = pd.DataFrame(arr, columns=[f"pixel_{i}" for i in range(arr.shape[1])])
df.insert(0, "filename", fns)
submission_path = os.path.join(WORK_DIR, "submission.csv")
df.to_csv(submission_path, index=False)
print("Saved:", submission_path, "| shape:", df.shape)
df.iloc[:3, :6]

In [ ]:
try:
    from google.colab import files; files.download(submission_path)
except Exception:
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [**radar-ioai-2025 제출 페이지**](https://www.kaggle.com/competitions/radar-ioai-2025/submit) 에 업로드.

Radar **모범답안2**(weighted-CE U-Net, 0.79→0.98)를 그대로 옮긴 버전. 핵심은 *배경 1 / 비배경 50 가중 CrossEntropy*
로 희소 클래스를 살리는 것. 더: 에폭↑, b(채널)↑, 증강, TTA. (대회: https://www.kaggle.com/competitions/radar-ioai-2025)